In [ ]:
from pprint import PrettyPrinter

import pandas as pd
import plotly.express as px
from country_converter import CountryConverter
from IPython.display import VimeoVideo
from pymongo import MongoClient


In [ ]:
VimeoVideo("733383823", h="d6228d4de1", width=600)


In [ ]:
VimeoVideo("733383369", h="4d221e7fb7", width=600)


In [ ]:
pp = PrettyPrinter(indent=2)
print("pp type:", type(pp))


In [ ]:
host = "192.252.53.2"


In [ ]:
VimeoVideo("733383007", h="13b2c716ac", width=600)


In [ ]:
client = MongoClient(host=host, port=27017)
print("client type:", type(client))


In [ ]:
pp.pprint(list(client.list_databases()))


In [ ]:
VimeoVideo("733382605", h="e0b87a5ff8", width=600)


In [ ]:
db = client["wqu-abtest"]
ds_app = db["ds-applicants"]
print("ds_app type:", type(ds_app))


In [ ]:
VimeoVideo("733382346", h="9da7d3d1d8", width=600)


In [ ]:
# Count documents in `ds_app`
n_documents = ds_app.count_documents({})
print("Num. documents in 'ds-applicants':", n_documents)


In [ ]:
VimeoVideo("733380658", h="a7988083f4", width=600)


In [ ]:
result = ds_app.find_one({})
print("result type:", type(result))
pp.pprint(result)


In [ ]:
VimeoVideo("733379562", h="8ffd2458e0", width=600)


In [ ]:
result = ds_app.aggregate(
    [
        {"$group": {"_id": "$countryISO2", "count": {"$count":{}}
                   }
        }
    ]
)
print("result type:", type(result))


In [ ]:
VimeoVideo("733376898", h="fc7f30e75a", width=600)


In [ ]:
df_nationality = pd.DataFrame(result).rename({"_id": "country_iso2"}, axis="columns")
print("df_nationality type:", type(df_nationality))
print("df_nationality shape", df_nationality.shape)
df_nationality.head()


In [ ]:
VimeoVideo("733373453", h="f8e954db9f", width=600)


In [ ]:
cc = CountryConverter()
df_nationality["country_name"] = cc.convert(
    df_nationality["country_iso2"], to="name_short"
)

print("df_nationality shape:", df_nationality.shape)
df_nationality.head()


In [ ]:
VimeoVideo("733372561", h="2659ff0dc7", width=600)


In [ ]:
# Create a horizontal bar chart
fig = px.bar(
    data_frame=df_nationality.tail(10),
    x="count",
    y="country_name",
    orientation="h",
    title="DS Applicants: Nationality"
)
# Set axis labels
fig.update_layout(xaxis_title="Frequency [count]", yaxis_title="country")

fig.show()


In [ ]:
VimeoVideo("733371952", h="a061e33ab8", width=600)


In [ ]:
df_nationality["count_pct"] = (
    df_nationality["count"] / df_nationality["count"].sum()
) * 100
print("df_nationality shape:", df_nationality.shape)
df_nationality.head()


In [ ]:
VimeoVideo("733371556", h="7cae7252a8", width=600)


In [ ]:
fig = px.bar(
    data_frame=df_nationality.tail(10),
    x="count_pct",
    y="country_name",
    orientation="h",
    title="DS Applicants: Nationality"
)
# Set axis labels
fig.update_layout(xaxis_title="Frequency [%]", yaxis_title="country")

fig.show()


In [ ]:
VimeoVideo("733370726", h="2b21ee76d2", width=600)


In [ ]:
df_nationality["country_iso3"] = cc.convert(df_nationality["country_iso2"], to="ISO3")
print("df_nationality shape:", df_nationality.shape)
df_nationality.head()


In [ ]:
VimeoVideo("733369606", h="73a380a6c6", width=600)


In [ ]:
def build_nat_choropleth():
    fig = px.choropleth(
        data_frame=df_nationality,
        locations="country_iso3",
        color="count_pct",
        projection="natural earth",
        color_continuous_scale=px.colors.sequential.Oranges,
        title="DS Applicants Nationality"
    )
    return fig

nat_fig = build_nat_choropleth()
print("nat_fig type:", type(nat_fig))
nat_fig.show()


In [ ]:
VimeoVideo("733367865", h="6e444cb810", width=600)


In [ ]:
result = ds_app.aggregate([
    {
        "$project": {
            "years": {
                "$dateDiff": {
                    "startDate": "$birthday",
                    "endDate": "$$NOW",
                    "unit": "year"
                }
            }
        }
    }
])

print("result type:", type(result))


In [ ]:
VimeoVideo("733367340", h="2b926b1e3a", width=600)


In [ ]:
ages = pd.DataFrame(result)["years"]

print("ages type:", type(ages))
print("ages shape:", ages.shape)
ages.head()


In [ ]:
VimeoVideo("733366740", h="bb14c884bb", width=600)


In [ ]:
def build_age_hist():
    # Create histogram of `ages`
    fig = px.histogram(
        x=ages, nbins=20, title="Distribution of DS Applicant Ages"
    )
    # Set axis labels
    fig.update_layout(
    xaxis_title="Age",
    yaxis_title="Frequency [count]"
)
    return fig

age_fig = build_age_hist()
print("age_fig type:", type(age_fig))
age_fig.show()


In [ ]:
VimeoVideo("733366435", h="c6d3a83830", width=600)


In [ ]:
result = ds_app.aggregate([
  {
    "$group": {
      "_id": "$highestDegreeEarned",
      "count": { "$count": {} }
    }
  }
])
print("result type:", type(result))


In [ ]:
VimeoVideo("733365459", h="5c14d30a9e", width=600)


In [ ]:
education = (
    pd.DataFrame(result)
    .rename({"_id": "highest_degree_earned"}, axis="columns")
    .set_index("highest_degree_earned")
    .squeeze()
)

print("education type:", type(education))
print("education shape:", education.shape)
education.head()


In [ ]:
VimeoVideo("733362518", h="90dd9a3394", width=600)


In [ ]:
def ed_sort(counts):
    """Sort array `counts` from highest to lowest degree earned."""
    degrees = [
        "High School or Baccalaureate",
        "Some College (1-3 years)",
        "Bachelor's degree",
        "Master's degree",
        "Doctorate (e.g. PhD)",
    ]
    mapping = {k: v for v, k in enumerate(degrees)}
    sort_order = [mapping[c] for c in counts]
    return sort_order


education.sort_index(key=ed_sort, inplace=True)
education


In [ ]:
VimeoVideo("733360047", h="b17fffc11b", width=600)


In [ ]:
def build_ed_bar():
    # Create bar chart
    fig = px.bar(
        x=education.values,
        y=education.index,
        orientation="h",
        title="DS Applicant Education Levels"
    )

    # Add axis labels
    fig.update_layout(
        xaxis_title="Frequency [count]",
        yaxis_title="Highest Degree Earned"
    )

    return fig


ed_fig = build_ed_bar()
print("ed_fig type:", type(ed_fig))
ed_fig.show()
